<a href="https://colab.research.google.com/github/shinhaan2009/E-coli-simulation/blob/main/%E1%84%80%E1%85%B2%E1%84%8C%E1%85%A6_%E1%84%89%E1%85%B5%E1%86%AF%E1%84%89%E1%85%B3%E1%86%B8%E1%84%87%E1%85%A9%E1%84%80%E1%85%A9%E1%84%89%E1%85%A5_%E1%84%92%E1%85%A1%E1%86%A8%E1%84%89%E1%85%A2%E1%86%BC%E1%84%8B%E1%85%AD%E1%86%BC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 특성 공학과 규제 — 실습 보고서

**인공지능 프로그래밍 과제 · 당뇨병 진행도 예측**

| | |
|---|---|
| **학번** | |
| **이름** | |
| **제출일** | |

---

## 이 보고서를 쓰는 법

이 노트북에는 두 종류의 할 일이 있습니다.

| 표시 | 무엇을 하는가 | 개수 |
|---|---|---|
| `【빈칸 N】` | **코드**의 `____` 부분을 채운다 | 8개 |
| **문제 N)** | **글로 답한다.** 답을 쓰는 칸이 바로 아래에 있다 | 17개 |

**문제 번호가 붙지 않은 곳은 답을 쓰지 않아도 됩니다.** 설명만 읽고 넘어가세요.

각 문제에는 몇 문장으로 쓰라는 지시가 붙어 있습니다. 길게 쓴다고 점수가
올라가지 않습니다.

### 제출

* 이 노트북 파일 하나만 제출합니다. 파일명: `학번_이름_규제보고서.ipynb`
* 제출 전에 반드시 `런타임 → 런타임 초기화 후 모든 셀 실행` (Colab) 또는
  `Kernel → Restart & Run All` 을 실행해, 처음부터 끝까지 오류 없이
  돌아가는지 확인하시오. **이것만으로 15점입니다.**
* Colab: `파일 → 다운로드 → .ipynb 다운로드`

### 채점 기준

| 항목 | 배점 |
|---|---|
| 재현성 (오류 없이 실행) | 15 |
| 빈칸 | 15 |
| **문제 답안의 질** | **50** |
| 가설 문제(4, 9)의 정직성 | 20 |

### 가설 문제에 대하여

**예측이 맞았는지는 채점하지 않습니다.** 틀린 예측을 솔직히 적고 왜 틀렸는지
잘 분석한 보고서가 만점입니다. 실제로 문제 4와 9는 **대부분 예측이 틀립니다.**
결과를 본 뒤에 가설을 고쳐 쓰면, 배점 20점을 스스로 버리는 것입니다.

---

## 이 과제의 질문

수업에서 우리는 농어의 무게를 예측하면서 **세 개의 벽**에 부딪혔습니다.

| | 무슨 일이 일어났는가 | 해결 |
|---|---|---|
| **1차 벽** | 특성 55개 > 샘플 42개 → 훈련 1.0 / 테스트 **−144.4** | 특성을 버리지 않고 **규제** |
| **2차 벽** | 특성마다 스케일이 달라 계수를 공정히 비교할 수 없다 | **표준화** |
| **3차 벽** | `alpha` 기본값 1이 최선이 아니다 | **튜닝** |

그리고 규칙을 다섯 개 얻었습니다.

> **규칙 1.** 훈련 점수가 만점에 가까우면 오히려 의심하라
> **규칙 2.** 규제를 걸기 전에는 반드시 표준화한다
> **규칙 3.** 과대적합은 ① 단순하게 만들거나 ② 자유를 제한(규제)해서 고친다
> **규칙 4.** 하이퍼파라미터는 기본값을 믿지 말고 직접 확인한다
> **규칙 5.** 최적 `alpha` 는 모델마다 다르다

> ### 그 규칙들은 농어에만 통하는 요령이었을까, 아니면 원리였을까?

**완전히 다른 데이터**로 같은 실험을 반복해 확인합니다.
당뇨병 환자 442명의 신체·혈액 정보로 **1년 뒤 질병 진행 정도**를 예측하는 문제입니다.

미리 말해 둡니다. **이 데이터는 농어처럼 얌전하지 않습니다.**
농어는 규제를 걸자 −144.4가 0.98로 완전히 되살아났습니다.
여기서는 그렇게 되지 않습니다. **그 불편함이 이 과제의 핵심입니다.**

---

## '답'이란 무엇인가

가장 많이 틀리는 부분입니다. 아래 두 예를 비교하시오.

### ❌ 이것은 결과를 다시 쓴 것입니다

> degree가 2일 때는 테스트 $R^2$ 가 0.4242였고 3일 때는 −55.94로 떨어졌다.
> 따라서 degree를 너무 높이면 안 된다는 것을 알 수 있다.

### ⭕ 이것이 답입니다

> degree=2에서 특성 65개로 훈련 0.6048 / 테스트 0.4242였는데, degree=3에서
> 특성이 285개가 되자 훈련은 0.9098로 올라간 반면 테스트는 −55.94가 되었다.
> 훈련만 오르고 테스트가 무너졌다는 것은 모델이 훈련 데이터의 노이즈까지
> 외웠다는 뜻이다. 농어에서도 특성 55개일 때 훈련 1.0 / 테스트 −144.4로
> 같은 모양이 나타났으므로, 이는 이 데이터만의 특징이 아니라 **특성 수가
> 늘어날 때 반복되는 현상**으로 보인다.

차이는 **숫자를 근거로 인과를 설명했는가**입니다.
"~를 알 수 있다", "~가 중요하다" 로 끝나는 문장은 답이 아닙니다.

---

# 0. 준비

수업에서 쓴 것과 거의 같습니다. 농어는 인터넷에서 CSV를 읽어 왔지만,
이 데이터는 사이킷런에 내장되어 있어 함수 하나로 불러옵니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes                 # 오늘의 데이터
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

In [ ]:
# scaled=False 는 '가공하지 않은 원래 값'으로 달라는 뜻입니다.
diabetes = load_diabetes(scaled=False)

target = diabetes.target

print('데이터 크기 (샘플 수, 특성 수):', diabetes.data.shape)
print('특성 이름:', diabetes.feature_names)
print()
print('타깃 평균    :', target.mean().round(2))
print('타깃 표준편차:', target.std().round(2))
print('타깃 범위    :', target.min(), '~', target.max())

특성의 뜻은 다음과 같습니다. `s1`~`s6` 은 혈액 검사 수치입니다.

| 이름 | 뜻 |
|---|---|
| age | 나이 |
| sex | 성별 (1 또는 2) |
| bmi | 체질량지수 |
| bp | 평균 혈압 |
| s1~s6 | 혈청 검사 6종 |

**타깃**은 1년 뒤 질병이 얼마나 진행되었는지를 나타내는 수치입니다.
클수록 많이 진행된 것입니다.

### 문제 1) 출력을 눈으로 읽고 아래를 채우시오.

* 샘플 수는 **____** 명, 특성은 **____** 개이다.
* 타깃의 평균은 **____**, 표준편차는 **____** 이다.

### 문제 2) 아무것도 학습하지 않고 **항상 평균만 답하는 모델**을 생각해 보시오. **2~3문장**

* 그 모델의 $R^2$ 는 몇 점인가? 수업에서 배운 $R^2$ 의 정의로 답하시오.
* 그렇다면 $R^2$ 가 **음수**라는 것은 무슨 뜻인가?

>
>

---

# 1부. 특성을 늘리다

## 1-1. 출발점 — 특성 10개 그대로

먼저 가공 없이, 있는 특성 10개를 그대로 써서 선형 회귀를 돌립니다.
**이 점수가 앞으로 계속 비교할 기준선입니다.** 잘 기억해 두시오.

In [ ]:
# 【빈칸 1】 특정 열이 아니라 전체 특성을 씁니다. diabetes 의 무엇을 넣어야 할까요?
train_full, test_full, train_target, test_target = train_test_split(
    diabetes.____, target, random_state=42)

print('훈련 세트:', train_full.shape, '/ 테스트 세트:', test_full.shape)

lr = LinearRegression()
lr.fit(train_full, train_target)

print()
print('훈련 세트 R^2 :', lr.score(train_full, train_target))
print('테스트 세트 R^2:', lr.score(test_full, test_target))

### 문제 3) 결과를 **1~2문장**으로 적으시오.

훈련 세트의 샘플 수(**____** 개)도 함께 적으시오. 뒤에서 계속 쓰는 숫자입니다.

>

---

## 1-2. 다항 특성으로 늘리기

수업에서 `PolynomialFeatures` 로 특성 3개를 9개, 55개로 늘렸습니다.
여기서는 특성 10개를 출발점으로 `degree` 를 1부터 4까지 올려 봅니다.

### 문제 4) **실행하기 전에** 예측하시오. (가설 문제)

`degree` 를 올리면 특성이 몇 개까지 늘어날 것 같습니까?
그리고 테스트 $R^2$ 는 어떻게 움직일 것 같습니까? **숫자로** 예측하고
근거를 **2~3문장**으로 쓰시오.

| degree | 특성 수 예상 | 테스트 $R^2$ 예상 |
|---|---|---|
| 2 | | |
| 3 | | |
| 4 | | |

**근거:**

>
>

In [ ]:
for degree in [1, 2, 3, 4]:
    # 【빈칸 2】 이번 반복의 차수로 다항 특성을 만드시오. 절편 자리(1)는 만들지 않습니다.
    poly = PolynomialFeatures(degree=____, include_bias=____)
    poly.fit(train_full)
    tr_poly = poly.transform(train_full)
    te_poly = poly.transform(test_full)

    lr = LinearRegression()
    lr.fit(tr_poly, train_target)

    print('degree=%d | 특성 %5d개 | 훈련 %8.4f | 테스트 %14.4f'
          % (degree, tr_poly.shape[1],
             lr.score(tr_poly, train_target),
             lr.score(te_poly, test_target)))

## 여기가 이 과제의 핵심입니다

### 문제 5) 위 출력을 표로 옮겨 적으시오.

| degree | 특성 수 | 훈련 $R^2$ | 테스트 $R^2$ |
|---|---|---|---|
| 1 | | | |
| 2 | | | |
| 3 | | | |
| 4 | | | |

* 훈련 세트의 샘플 수는 **____** 개이다.
* 테스트 $R^2$ 가 **음수**로 바뀌는 것은 degree = **____** 일 때이다.
  그때 특성 수는 **____** 개이다.
* 특성 수가 샘플 수를 **처음으로 넘어서는** 것은 degree = **____** 일 때이다.
  그때 훈련 $R^2$ 는 **____** 이다.

### 문제 6) 위에서 찾은 두 지점이 **서로 다릅니다.** 이를 **4~6문장**으로 설명하시오. 배점이 가장 큰 문항입니다.

수업에서는 "특성 55개 > 샘플 42개" 였고, 특성이 샘플을 넘어선 바로 그 지점에서
테스트 점수가 무너졌습니다. 그래서 두 사건이 **같은 지점**에서 일어났습니다.
그런데 이번 데이터에서는 **음수가 먼저 나타나고, 특성이 샘플을 넘는 것은 그 뒤**입니다.

다음을 반드시 포함하시오.

* 특성이 샘플보다 많아지면 훈련 $R^2$ 가 왜 1.0이 되는가
* 그런데 특성이 샘플보다 **적은데도** 테스트가 무너진 것은 무엇을 뜻하는가
* 그렇다면 "특성 수 > 샘플 수"는 과대적합의 **원인**인가, 아니면 여러 경우 중
  **가장 극단적인 한 경우**인가? 근거를 들어 답하시오.

>
>
>
>

---

# 2부. 원인은 계수에 있다

수업에서 확인했습니다. 과대적합의 정체는 **터무니없이 커진 계수**였습니다.
농어에서는 계수 하나가 2만을 넘었죠.

이제 `degree=3` 으로 고정하고 (특성 285개) 계수를 직접 들여다봅니다.

In [ ]:
# 【빈칸 3】 degree=3 으로 다항 특성을 만드시오.
poly = PolynomialFeatures(degree=____, include_bias=False)
poly.fit(train_full)

train_poly = poly.transform(train_full)
test_poly  = poly.transform(test_full)

lr = LinearRegression()
lr.fit(train_poly, train_target)

print('특성 수          :', train_poly.shape[1])
print('계수 절댓값 최대  :', np.abs(lr.coef_).max())
print('계수 절댓값 평균  :', np.abs(lr.coef_).mean())
print()
print('테스트 세트 R^2  :', lr.score(test_poly, test_target))

### 문제 7) 계수의 크기를 **2~4문장**으로 해석하시오.

다음을 반드시 포함하시오.

* 계수 절댓값의 최댓값을 숫자로 인용할 것
* 타깃(질병 진행도)의 범위는 25~346이었다. 계수의 크기와 비교하면 무엇이 이상한가
* 계수가 크면 왜 새로운 데이터에서 예측이 무너지는가

>
>
>

---

# 3부. 규제로 되살리기

## 3-1. 먼저 표준화

특성 285개 중에는 `bmi^3` 처럼 세 번 곱한 것도 있습니다.
값의 크기가 특성마다 천차만별이라, 이대로는 "계수가 크다"를 공정하게
판단할 수 없습니다.

In [ ]:
ss = StandardScaler()

# 【빈칸 4】 변환기는 어떤 세트로 fit 해야 할까요?
ss.fit(____)

train_scaled = ss.transform(train_poly)
test_scaled  = ss.transform(test_poly)

print('표준화 전 첫 샘플의 앞 3개:', train_poly[0][:3].round(2))
print('표준화 후 첫 샘플의 앞 3개:', train_scaled[0][:3].round(2))

### 문제 8) 위 빈칸에 `test_poly` 를 넣으면 안 되는 이유를 **2~3문장**으로 설명하시오.

수업에서 배운 **데이터 누출**의 관점에서 답하시오.

>
>

---

## 3-2. 릿지

### 문제 9) **실행하기 전에** 예측하시오. (가설 문제)

지금 이 모델의 테스트 $R^2$ 는 **−55.94** 입니다.

농어에서는 규제를 걸자 −144.4가 **0.9828** 로 완전히 되살아났습니다.
이 데이터에 릿지를 걸면 테스트 $R^2$ 가 어디까지 회복될 것 같습니까?
**숫자로** 예측하고 근거를 **2~3문장**으로 쓰시오.

**나의 예측:** $R^2 \approx$ ______

>
>

In [ ]:
# 규제 없는 모델과 릿지의 계수 크기를 비교합니다.
lr = LinearRegression()
lr.fit(train_scaled, train_target)

ridge = Ridge(alpha=10)
ridge.fit(train_scaled, train_target)

print('규제 없음 → 계수 절댓값 최대: %15.1f' % np.abs(lr.coef_).max())
print('릿지      → 계수 절댓값 최대: %15.1f' % np.abs(ridge.coef_).max())

In [ ]:
# 【빈칸 5】 이번 반복의 alpha 값으로 릿지 모델을 만드시오.
alpha_list = [0.01, 0.1, 1, 10, 100, 1000]
train_scores = []
test_scores  = []

for alpha in alpha_list:
    ridge = Ridge(alpha=____)
    ridge.fit(train_scaled, train_target)
    train_scores.append(ridge.score(train_scaled, train_target))
    test_scores.append(ridge.score(test_scaled, test_target))

plt.plot(np.log10(alpha_list), train_scores, label='train')
plt.plot(np.log10(alpha_list), test_scores, label='test')
plt.xlabel('log10(alpha)')
plt.ylabel('R^2')
plt.title('Ridge')
plt.legend()
plt.show()

for a, tr, te in zip(alpha_list, train_scores, test_scores):
    print('alpha =', a, '\t훈련', round(tr, 4), '\t테스트', round(te, 4))

### 문제 10) 위 결과에서 아래를 채우시오.

* 규제 없는 모델의 계수 절댓값 최대: **____**
* 릿지($\alpha=10$)의 계수 절댓값 최대: **____**
* 테스트 $R^2$ 가 가장 높은 $\alpha$ = **____** (그때 $R^2$ = **____**)
* 규제 없이 얻었던 테스트 $R^2$ 는 **____** 였다.

### 문제 11) 예측(문제 9)과 실제 결과를 비교하여 **4~6문장**으로 쓰시오.

다음을 반드시 포함하시오.

* 계수가 몇 배나 줄었는가 (숫자로)
* 농어는 0.98까지 회복했는데 이 데이터는 그러지 못했다. 규제가 실패한 것인가?
* **1부에서 구한 기준선**(특성 10개 그대로 쓴 선형 회귀)과 비교하면 어떤가?

>
>
>
>

---

# 4부. 라쏘

릿지와 딱 한 곳만 다릅니다.

| | 벌점 항 | 계수를 어떻게 하나 |
|---|---|---|
| **릿지 (Ridge)** | α × Σ(계수)² — **제곱** | 0에 **가깝게** 만든다 |
| **라쏘 (Lasso)** | α × Σ\|계수\| — **절댓값** | 아예 **0으로** 만들 수 있다 |

릿지와 **같은 방법으로** `alpha` 를 튜닝합니다. 모델 이름만 바꾸면 됩니다.

> `max_iter` 는 계산 반복 횟수입니다. 라쏘는 답을 한 번에 계산하지 못하고
> 조금씩 고쳐가며 찾기 때문에, 넉넉히 주지 않으면 "수렴하지 않았다"는
> 경고가 뜹니다. 특성이 285개라 이번에는 아주 크게 잡았습니다.
> **이 셀은 실행에 10~30초쯤 걸립니다.** 멈춘 것이 아니니 기다리시오.

In [ ]:
# 【빈칸 6】 릿지 코드에서 모델 이름만 바꾸면 됩니다. (max_iter=500000 도 함께)
lasso_train_scores = []
lasso_test_scores  = []
n_used = []

for alpha in alpha_list:
    lasso = ____(alpha=alpha, max_iter=500000)
    lasso.fit(train_scaled, train_target)

    lasso_train_scores.append(lasso.score(train_scaled, train_target))
    lasso_test_scores.append(lasso.score(test_scaled, test_target))
    n_used.append(np.sum(lasso.coef_ != 0))

plt.plot(np.log10(alpha_list), lasso_train_scores, label='train')
plt.plot(np.log10(alpha_list), lasso_test_scores, label='test')
plt.xlabel('log10(alpha)')
plt.ylabel('R^2')
plt.title('Lasso')
plt.legend()
plt.show()

for a, tr, te, n in zip(alpha_list, lasso_train_scores, lasso_test_scores, n_used):
    print('alpha =%6s\t훈련' % a, round(tr, 4), '\t테스트', round(te, 4),
          '\t쓰인 특성', n, '개')

### 문제 12) 위 표에서 아래를 찾아 채우시오.

* 테스트 $R^2$ 가 가장 높은 $\alpha$ = **____** (그때 $R^2$ = **____**, 쓰인 특성 **____** 개)
* $\alpha = 100$ 일 때 쓰인 특성 수 = **____** 개, 훈련 $R^2$ = **____**, 테스트 $R^2$ = **____**

### 문제 13) $\alpha = 100$ 에서 벌어진 일을 **3~5문장**으로 설명하시오.

훈련 $R^2$ 가 정확히 **0.0** 입니다. 우연이 아닙니다.

다음을 반드시 포함하시오.

* 쓰인 특성이 0개라는 것은 예측식이 어떤 모양이 되었다는 뜻인가
* 그 모델은 입력과 상관없이 무슨 값을 답하는가
* 그 값이 **문제 2**에서 생각한 모델과 같은가? 그래서 $R^2$ 가 0.0인가?

>
>
>

In [ ]:
# 【빈칸 7】 위에서 찾은 최적의 alpha 값을 넣으시오.
lasso = Lasso(alpha=____, max_iter=500000)
lasso.fit(train_scaled, train_target)

print('훈련 세트 R^2 :', lasso.score(train_scaled, train_target))
print('테스트 세트 R^2:', lasso.score(test_scaled, test_target))
print()
print('전체 특성 수       :', len(lasso.coef_))
print('계수가 0인 특성 수 :', np.sum(lasso.coef_ == 0))
print('실제로 쓰인 특성 수:', np.sum(lasso.coef_ != 0))

라쏘가 **남긴 특성이 무엇인지** 이름으로 확인해 봅시다.
`get_feature_names_out()` 에 원래 특성 이름을 넣어 주면 `x0` 대신
`bmi`, `bmi s5` 처럼 읽을 수 있는 이름으로 나옵니다.

In [ ]:
# 【빈칸 8】 계수가 0이 '아닌' 것만 골라내시오. (== 인지 != 인지 생각할 것)
names = poly.get_feature_names_out(diabetes.feature_names)

result = pd.DataFrame({'특성': names, '계수': lasso.coef_})
result[result['계수'] ____ 0]

### 문제 14) 남은 특성들을 **4~6문장**으로 해석하시오.

다음을 반드시 포함하시오.

* 285개 중 몇 개가 버려지고 몇 개가 남았는가 (숫자)
* 남은 특성들의 **이름**을 보시오. 반복해서 등장하는 원래 특성이 있는가?
  있다면 그것은 무엇을 시사하는가
* 그렇게 많이 버렸는데도 테스트 점수가 릿지와 비슷한 이유는 무엇이라고 생각하는가

>
>
>
>

### 문제 15) 릿지와 라쏘 중 이 데이터에는 어느 쪽이 더 적합해 보이는가? **3~5문장**

"둘 다 장단점이 있다"로 끝나면 답이 아닙니다. **하나를 고르고 이유를 대시오.**
테스트 점수뿐 아니라, 이 모델을 **의사에게 설명해야 하는 상황**도 함께 고려하시오.

>
>
>

---

# 5부. 종합

### 문제 16) 아래 표를 채우시오.

| | 모델 | 특성 수 | 테스트 $R^2$ |
|---|---|---|---|
| 기준선 | 선형 회귀 (원래 특성 10개) | 10 | |
| 1부 | 다항 회귀 (degree=3, 규제 없음) | 285 | |
| 3부 | 릿지 (degree=3, 최적 $\alpha$) | 285 | |
| 4부 | 라쏘 (degree=3, 최적 $\alpha$) | | |

---

### 문제 17) 위 표를 보면 **불편한 사실**이 하나 있습니다. **5~8문장**

특성을 10개에서 285개로 늘리고, 표준화하고, 규제를 걸고, `alpha` 까지
튜닝했습니다. 그런데 최종 점수는 **맨 처음 특성 10개를 그냥 썼을 때와
거의 같습니다.**

* 그렇다면 이 모든 과정은 **헛수고**였습니까?
* 만약 헛수고가 아니라면, 우리가 **얻은 것**은 무엇입니까?
  (힌트: 처음에 우리는 285개 중 어느 것이 쓸모 있는지 몰랐습니다)
* 만약 헛수고에 가깝다면, 이 데이터에서는 애초에 **무엇을 했어야** 합니까?

어느 쪽으로 답해도 좋습니다. **근거를 대시오.**

>
>
>
>
>

### 문제 18) 이 과제의 질문에 답하시오. **5~8문장**

> 수업에서 얻은 규칙들은 농어에만 통하는 요령이었을까, 아니면 원리였을까?

아래 규칙을 **각각** 짚어 답하시오. 그대로 통한 것, 겉모습은 달랐지만
원리는 같았던 것을 구분하시오.

* 규칙 1 (훈련 점수가 만점에 가까우면 의심하라)
* 규칙 3 (과대적합은 단순화하거나 규제해서 고친다)
* 규칙 5 (최적 `alpha` 는 모델마다 다르다)
  — 이 데이터에서 릿지와 라쏘의 최적 `alpha` 는 어땠는가? 농어와 같았는가?

>
>
>
>
>

### 문제 19) 농어의 최고 $R^2$ 는 0.98이었고, 이 데이터는 0.5 안팎입니다. **4~6문장**

* 이것은 우리가 모델을 잘못 골랐기 때문인가, 아니면 다른 이유가 있는가?
* 두 문제(농어 무게 예측 / 질병 진행도 예측)의 성격이 어떻게 다른지 생각해 보시오.
* $R^2 = 0.5$ 인 모델은 쓸모없는 모델인가? **문제 2**에서 생각한 기준과 비교해 답하시오.

>
>
>
>

### 문제 20) 이 실험을 하면서 새로 생긴 의문을 **하나 이상** 적으시오.

답을 몰라도 됩니다. 좋은 질문 자체가 점수입니다.

>

---

## 제출 전 점검

- [ ] `런타임 초기화 후 모든 셀 실행` 을 했고 오류가 없다
- [ ] 빈칸 8개를 모두 채웠다
- [ ] **문제 1)부터 20)까지 빠짐없이 답했다**
- [ ] 가설 문제(4, 9)를 결과보다 **먼저** 썼다
- [ ] 답에 구체적인 숫자가 들어 있다 ("~를 알 수 있다"로 끝나지 않는다)
- [ ] 파일명이 `학번_이름_규제보고서.ipynb` 이다